In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, classification_report, matthews_corrcoef

# File paths
DATA_PATH = 'data.csv'
MODEL_PATH = 'xgb_model.pkl'
FEATURES_PATH = 'feature_names.pkl'

# Load dataset
data = pd.read_csv(DATA_PATH)

# Feature-target separation
X = data.drop(columns=['Target'])
y = data['Target']

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.4, random_state=42, stratify=y
)

# Balance classes in training set with SMOTE
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

# Define XGBoost model
model = XGBClassifier(
    objective='multi:softprob',
    num_class=len(np.unique(y)),
    use_label_encoder=False,
    eval_metric='mlogloss',
    random_state=42
)

# Hyperparameter tuning
param_grid = {
    'n_estimators': [96],
    'max_depth': [6],
    'learning_rate': [0.1],
    'subsample': [0.8],
    'colsample_bytree': [0.8],
    'min_child_weight': [100],
}

grid_search = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    scoring='accuracy',
    cv=5,
    n_jobs=-1,
    verbose=1
)

# Fit model
grid_search.fit(X_train_res, y_train_res)

# Get best model
best_model = grid_search.best_estimator_
best_params = grid_search.best_params_
print("Best Hyperparameters:", best_params)

# Save model and feature names
joblib.dump(best_model, MODEL_PATH)
joblib.dump(X.columns.tolist(), FEATURES_PATH)
print("Model and feature names saved successfully.")

# Predict on test data
y_pred = best_model.predict(X_test)

# Evaluate
accuracy = accuracy_score(y_test, y_pred)
mcc = matthews_corrcoef(y_test, y_pred)
report = classification_report(y_test, y_pred)

print(f"\nAccuracy: {accuracy:.4f}")
print(f"Matthews Correlation Coefficient (MCC): {mcc:.4f}")
print("\nClassification Report:")
print(report)


Fitting 5 folds for each of 1 candidates, totalling 5 fits


/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [02:05:29] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


Best Hyperparameters: {'colsample_bytree': 0.8, 'learning_rate': 0.1, 'max_depth': 6, 'min_child_weight': 100, 'n_estimators': 96, 'subsample': 0.8}
Model and feature names saved successfully.

Accuracy: 0.9573
Matthews Correlation Coefficient (MCC): 0.9434

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      2150
           1       1.00      1.00      1.00      2179
           2       0.95      0.88      0.91      2159
           3       0.89      0.95      0.92      2128

    accuracy                           0.96      8616
   macro avg       0.96      0.96      0.96      8616
weighted avg       0.96      0.96      0.96      8616

